# Deployed Model Testing with Batch Predictions

This notebook demonstrates how to test a deployed model using its scoring endpoint with batch predictions.

## Workflow
1. Setup and Configuration
2. Load Environment Variables (including Deployment Endpoint URL)
3. Generate Test Samples
4. Send Batch Prediction Request
5. Analyze Results

## 1. Setup and Configuration

In [5]:
# Import required libraries
import os
import json
import numpy as np
from datetime import datetime, timezone, timedelta
from importlib.metadata import version
import warnings
warnings.filterwarnings('ignore')

# Set timezone to UTC+8 (Asia/Taipei)
TZ_UTC8 = timezone(timedelta(hours=8))

# Get package versions
np_version = version('numpy')
watsonx_version = version('ibm-watsonx-ai')

print("="*80)
print("PACKAGE VERSIONS")
print("="*80)
print(f"✅ NumPy version: {np_version}")
print(f"✅ IBM watsonx.ai version: {watsonx_version}")
print("\n✅ All libraries imported successfully")

PACKAGE VERSIONS
✅ NumPy version: 1.26.4
✅ IBM watsonx.ai version: 1.5.3

✅ All libraries imported successfully


## 2. Load Environment Variables

This cell loads the deployment model name, space ID, and API key from environment variables.

In [7]:
# Load credentials from environment variables
DEPLOYMENT_NAME = os.getenv('DEPLOYMENT_NAME')
SPACE_ID = os.getenv('SPACE_ID')
WX_API_KEY = os.getenv('WX_API_KEY')

print("="*80)
print("ENVIRONMENT VARIABLES CHECK")
print("="*80)
print(f"DEPLOYMENT_NAME: {'✅ Set' if DEPLOYMENT_NAME else '❌ Not set'}")
print(f"SPACE_ID: {'✅ Set' if SPACE_ID else '❌ Not set'}")
print(f"WX_API_KEY: {'✅ Set' if WX_API_KEY else '❌ Not set'}")
print("="*80)

if not all([DEPLOYMENT_NAME, SPACE_ID, WX_API_KEY]):
    print("\n⚠️ WARNING: Required environment variables are missing!")
    print("Please set: DEPLOYMENT_NAME, SPACE_ID, WX_API_KEY")
    print("\nExample:")
    print("export DEPLOYMENT_NAME='buy_model_exchange'")
    print("export SPACE_ID='your-space-id'")
    print("export WX_API_KEY='your-api-key'")
else:
    print("\n✅ All required environment variables are set!")
    print(f"\n📍 Model Name: {DEPLOYMENT_NAME}")
    print(f"📍 Space ID: {SPACE_ID}")

ENVIRONMENT VARIABLES CHECK
DEPLOYMENT_NAME: ✅ Set
SPACE_ID: ✅ Set
WX_API_KEY: ✅ Set

✅ All required environment variables are set!

📍 Model Name: savedmodel_1231_deployment
📍 Space ID: 165e735f-11f6-4879-a6fd-99cfecf44f37


## 3. Define Input Configuration

Based on the model architecture, we define the configuration for all 32 inputs.

In [8]:
# Input configuration from the model
inputs_config = [
    # BLOCK_MD_BRN_CONTACTS
    ("input_x_o_MD_BRN_CONTACTS", (16,), "float"),
    ("input_x_c_1_MD_BRN_CONTACTS", (1,), "int", 12),
    ("input_x_c_2_MD_BRN_CONTACTS", (1,), "int", 6),
    ("input_x_c_3_MD_BRN_CONTACTS", (1,), "int", 3),
    ("input_x_c_4_MD_BRN_CONTACTS", (1,), "int", 6),
    ("input_x_c_5_MD_BRN_CONTACTS", (1,), "int", 7),
    ("input_x_c_6_MD_BRN_CONTACTS", (1,), "int", 10),
    ("input_x_c_7_MD_BRN_CONTACTS", (1,), "int", 104),
    ("input_x_c_8_MD_BRN_CONTACTS", (1,), "int", 8),
    
    # Time series
    ("input_x_t_MD_CASH_FLOW", (12, 22), "float"),
    ("input_x_t_MD_CUS_EXCH", (12, 12), "float"),
    ("input_x_t_MD_CUS_LOAN", (12, 24), "float"),
    
    # BLOCK_MD_CUS_RISK
    ("input_x_o_MD_CUS_RISK", (1,), "float"),
    ("input_x_c_1_MD_CUS_RISK", (1,), "int", 7),
    ("input_x_c_2_MD_CUS_RISK", (1,), "int", 6),
    ("input_x_c_3_MD_CUS_RISK", (1,), "int", 5),
    ("input_x_c_4_MD_CUS_RISK", (1,), "int", 6),
    ("input_x_c_5_MD_CUS_RISK", (1,), "int", 6),
    ("input_x_c_6_MD_CUS_RISK", (1,), "int", 5),
    ("input_x_c_7_MD_CUS_RISK", (1,), "int", 7),
    ("input_x_c_8_MD_CUS_RISK", (1,), "int", 5),
    
    # BLOCK_MD_CUS_STYLE
    ("input_x_o_MD_CUS_STYLE", (35,), "float"),
    ("input_x_c_1_MD_CUS_STYLE", (1,), "int", 3),
    ("input_x_c_2_MD_CUS_STYLE", (1,), "int", 5),
    ("input_x_c_3_MD_CUS_STYLE", (1,), "int", 113),
    ("input_x_c_4_MD_CUS_STYLE", (1,), "int", 112),
    ("input_x_c_5_MD_CUS_STYLE", (1,), "int", 7),
    ("input_x_c_6_MD_CUS_STYLE", (1,), "int", 23),
    ("input_x_c_7_MD_CUS_STYLE", (1,), "int", 171),
    
    # More time series
    ("input_x_t_MD_DEP_AUM", (12, 12), "float"),
    ("input_x_t_MD_NEW_INVST", (12, 21), "float"),
    ("input_x_t_MD_PDH_AUM", (12, 30), "float"),
]

print(f"✅ Configured {len(inputs_config)} input features")

✅ Configured 32 input features


## 4. Generate Test Samples

In [9]:
def generate_test_sample(seed=42):
    """Generate a single test sample"""
    np.random.seed(seed)
    sample_data = {}
    
    for config in inputs_config:
        name = config[0]
        shape = config[1]
        dtype = config[2]
        
        if dtype == "int":
            vocab_size = config[3]
            data = np.random.randint(0, vocab_size, size=shape).tolist()
        else:
            data = np.random.randn(*shape).tolist()
        
        sample_data[name] = data
    
    return sample_data

# Generate multiple test samples
num_test_samples = 5
print(f"\n{'='*80}")
print(f"Generating {num_test_samples} test samples...")
print(f"{'='*80}")

test_samples = [generate_test_sample(seed=100+i) for i in range(num_test_samples)]

print(f"✅ Generated {len(test_samples)} test samples")
print(f"✅ Each sample has {len(test_samples[0])} features")


Generating 5 test samples...
✅ Generated 5 test samples
✅ Each sample has 32 features


## 5. Prepare Scoring Payload

In [10]:
print(f"\n{'='*80}")
print("Preparing Scoring Payload")
print(f"{'='*80}")

# Format for scoring - batch all samples together
input_data_list = []
for config in inputs_config:
    name = config[0]
    batch_values = [sample[name] for sample in test_samples]
    input_data_list.append({
        "id": name,
        "values": batch_values
    })

scoring_payload = {
    "input_data": input_data_list
}

print(f"✅ Scoring payload prepared")
print(f"   - Number of inputs: {len(input_data_list)}")
print(f"   - Batch size: {num_test_samples}")
print(f"\nPayload structure (first input):")
print(f"   Input ID: {input_data_list[0]['id']}")
print(f"   Number of samples: {len(input_data_list[0]['values'])}")


Preparing Scoring Payload
✅ Scoring payload prepared
   - Number of inputs: 32
   - Batch size: 5

Payload structure (first input):
   Input ID: input_x_o_MD_BRN_CONTACTS
   Number of samples: 5


## 6. Connect to watsonx.ai

#### CP4D

In [ ]:
# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

username = 'PASTE YOUR USERNAME HERE'
url = 'PASTE THE PLATFORM URL HERE'

print("✅ Configuration loaded")

In [ ]:
from ibm_watsonx_ai import Credentials, APIClient

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.1"
)

client = APIClient(credentials)
client.set.default_space(SPACE_ID)

print("="*80)
print("watsonx.ai Connection")
print("="*80)
print(f"✅ Connected to: {credentials.url}")
print(f"✅ Using Space ID: {SPACE_ID}")
print(f"✅ Client version: {client.version}")

#### IBM Cloud

In [ ]:
# from ibm_watsonx_ai import APIClient
# from ibm_watsonx_ai import Credentials

# # Connect to watsonx.ai
# credentials = Credentials(
#     url="https://us-south.ml.cloud.ibm.com",  # Replace with your region
#     api_key=WX_API_KEY
# )

# client = APIClient(credentials)
# client.set.default_project(PROJECT_ID)

# print("="*80)
# print("watsonx.ai Connection")
# print("="*80)
# print(f"✅ Connected to: {credentials.url}")
# print(f"✅ Project ID: {PROJECT_ID}")
# print(f"✅ Client version: {client.version}")

watsonx.ai Connection
✅ Connected to: https://us-south.ml.cloud.ibm.com
✅ Project ID: c4d2e318-f9d4-47c7-9482-f313463b22ea
✅ Client version: 1.5.3


## 7. Send Batch Prediction Request

In [ ]:
# Find deployment by model name
print(f"\n🔍 Searching for deployment with model name: {DEPLOYMENT_NAME}")

deployments = client.deployments.get_details()
deployment_id = None

for deployment in deployments['resources']:
    # Check if deployment name matches the model name
    deployment_name = deployment['metadata'].get('name', '')
    if deployment_name == DEPLOYMENT_NAME:
        deployment_id = deployment['metadata']['id']
        print(f"✅ Found deployment: {deployment_name}")
        print(f"✅ Deployment ID: {deployment_id}")
        break

if not deployment_id:
    print(f"\n❌ No deployment found with model name: {DEPLOYMENT_NAME}")
    print(f"Available deployments in space:")
    for deployment in deployments['resources'][:5]:
        print(f"  - {deployment['entity'].get('name', 'N/A')}")
    predictions = None
else:
    print(f"\n📊 Sending {num_test_samples} samples for scoring...")
    
    # Send prediction request using watsonx.ai SDK
    try:
        predictions = client.deployments.score(deployment_id, scoring_payload)
        print(f"\n✅ Scoring successful!")
    except Exception as e:
        print(f"\n❌ Scoring failed: {str(e)}")
        predictions = None

🔍 Searching for deployment with model name: savedmodel_1231_deployment
✅ Found deployment: savedmodel_1231_deployment
✅ Deployment ID: 84716fe3-b335-43c1-8411-5a8ce49e4486

📊 Sending 5 samples for scoring...

✅ Scoring successful!


## 8. Analyze Prediction Results

In [14]:
if predictions:
    print(f"\n{'='*80}")
    print("PREDICTION RESULTS")
    print(f"{'='*80}")
    
    import json
    print(json.dumps(predictions, indent=2, ensure_ascii=False))
else:
    print("❌ No predictions to display")


PREDICTION RESULTS
{
  "predictions": [
    {
      "id": "y_layers_2",
      "values": [
        [
          0.025820082053542137,
          0.03412525728344917,
          0.0011314157163724303,
          0.00920176412910223,
          0.0002816423075273633,
          0.0904655009508133,
          0.10830073803663254,
          0.13243147730827332,
          0.04879748076200485
        ],
        [
          0.004450659733265638,
          0.011267977766692638,
          0.00010887669486692175,
          0.0012351153418421745,
          0.00028459931490942836,
          0.006262253969907761,
          0.01695517636835575,
          0.015413716435432434,
          0.011043784208595753
        ],
        [
          0.0016264168079942465,
          0.009535297751426697,
          7.168121373979375e-05,
          0.0004189173341728747,
          3.5734377888729796e-05,
          0.002130347304046154,
          0.011342484503984451,
          0.008261310867965221,
          0.00823240727

## 9. Save Results (Optional)

In [15]:
if predictions:
    # Generate timestamp
    timestamp = datetime.now(TZ_UTC8).strftime("%Y%m%d_%H%M%S")
    results_filename = f"prediction_results_{timestamp}.json"
    
    # Save results
    with open(results_filename, 'w') as f:
        json.dump(predictions, f, indent=2)
    
    print(f"\n{'='*80}")
    print("Results Saved")
    print(f"{'='*80}")
    print(f"✅ Results saved to: {results_filename}")
    print(f"✅ File size: {os.path.getsize(results_filename) / 1024:.2f} KB")
else:
    print(f"\n⚠️ No results to save")


Results Saved
✅ Results saved to: prediction_results_20260326_124505.json
✅ File size: 5.62 KB


## 10. Summary

In [16]:
print(f"\n{'='*80}")
print("MODEL TESTING SUMMARY")
print(f"{'='*80}")

print(f"\n✅ Test Configuration:")
print(f"   - Deployment Name: {DEPLOYMENT_NAME}")
print(f"   - Space ID: {SPACE_ID}")
if 'deployment_id' in locals() and deployment_id:
    print(f"   - Deployment ID: {deployment_id}")
print(f"   - Test samples: {num_test_samples}")
print(f"   - Input features: {len(inputs_config)}")

if predictions:
    print(f"\n✅ Test Results:")
    print(f"   - Status: Success")
    print(f"   - Predictions received: {len(predictions.get('predictions', []))}")
    print(f"   - Results saved: {results_filename if 'results_filename' in locals() else 'N/A'}")
else:
    print(f"\n❌ Test Results:")
    print(f"   - Status: Failed")
    print(f"   - Please check model name, space ID, and API key")

print(f"\n{'='*80}")
print("✅ Model Testing Complete!")
print(f"{'='*80}")


MODEL TESTING SUMMARY

✅ Test Configuration:
   - Deployment Name: savedmodel_1231_deployment
   - Space ID: 165e735f-11f6-4879-a6fd-99cfecf44f37
   - Deployment ID: 84716fe3-b335-43c1-8411-5a8ce49e4486
   - Test samples: 5
   - Input features: 32

✅ Test Results:
   - Status: Success
   - Predictions received: 3
   - Results saved: prediction_results_20260326_124505.json

✅ Model Testing Complete!


## Summary

This notebook has demonstrated:
1. ✅ Loading deployment endpoint URL from environment variables
2. ✅ Generating test samples matching the model's input schema
3. ✅ Preparing batch prediction payload
4. ✅ Sending prediction request to deployed model
5. ✅ Analyzing and displaying results
6. ✅ Saving results to JSON file

**Key Points:**
- The deployment endpoint URL must be set as an environment variable
- Batch predictions allow testing multiple samples in a single request
- Results are automatically saved with timestamp for record keeping

**Next Steps:**
- Analyze prediction quality and model performance
- Compare predictions with expected outputs
- Monitor model behavior over time
- Implement automated testing pipelines